# Полный from-scratch SI VACV pipeline — BSC-40 + structural SAR

Все 30 LightGBM-моделей обучаются заново на исходном train. Готовые public
submissions, OOF и сохранённые модели не загружаются. Публичные ChEMBL/PMC/
PubChem/Wiley-измерения и BSC-40 восстанавливаются внутри notebook.

Последний слой — явные структурные SAR-эвристики из открытого решения
https://github.com/Velsaf-dev/molecular-ap-model. Это не экспериментальные
SI-метки и не чистая supervised ML. Сопоставление выполняется только по
SMILES/SMARTS. Официальный результат этой версии, сообщённый пользователем:
Kaggle MSE 82 826.34914.


In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, Descriptors, MACCSkeys
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.*")
SEEDS = [42, 13, 79]
N_FOLDS = 5
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "data/molecular_activity").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data/molecular_activity"
OUT = Path.cwd()
print("Project root:", ROOT)
print("Output:", OUT)

In [ ]:
tr = pd.read_csv(DATA / "train.csv")
te = pd.read_csv(DATA / "test.csv")
y = tr["SI VACV"].to_numpy(float)
y[tr["IC50_uM_VACV"].isna().to_numpy()] = 0.0
print("train/test:", tr.shape, te.shape, "| zeros:", int((y == 0).sum()), "| max SI:", y.max())

def std_mol(s):
    m = Chem.MolFromSmiles(str(s))
    try:
        m = rdMolStandardize.FragmentParent(m)
        m = Chem.MolFromSmiles(Chem.MolToSmiles(m))
    except Exception:
        pass
    return m

def key14(m):
    ik = Chem.MolToInchiKey(m)
    return ik[:14] if ik else ""

def simple_key14(s):
    m = Chem.MolFromSmiles(str(s))
    return key14(m) if m is not None else ""

def scaffold(m):
    try:
        sc = MurckoScaffold.GetScaffoldForMol(m)
        if sc.GetNumAtoms() == 0:
            return "ACYCLIC"
        return Chem.MolToSmiles(MurckoScaffold.MakeScaffoldGeneric(sc), isomericSmiles=False)
    except Exception:
        return "ACYCLIC"

def features(mols):
    rows = []
    for m in mols:
        fp = np.zeros(2048, np.uint8)
        DataStructs.ConvertToNumpyArray(AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048), fp)
        maccs = np.zeros(167, np.uint8)
        DataStructs.ConvertToNumpyArray(MACCSkeys.GenMACCSKeys(m), maccs)
        desc = [Descriptors.MolWt(m), Descriptors.MolLogP(m), Descriptors.TPSA(m),
                Descriptors.NumHDonors(m), Descriptors.NumHAcceptors(m), Descriptors.NumRotatableBonds(m),
                Descriptors.RingCount(m), Descriptors.FractionCSP3(m), Descriptors.HeavyAtomCount(m)]
        rows.append(np.concatenate([fp, maccs, desc]))
    return np.asarray(rows, dtype=np.float32)

tr_mols = [std_mol(s) for s in tr.SMILES]
te_mols = [std_mol(s) for s in te.SMILES]
X = features(tr_mols)
Xt = features(te_mols)
tr_keys = np.array([key14(m) for m in tr_mols])
te_keys = np.array([key14(m) for m in te_mols])
tr_scafs = np.array([scaffold(m) for m in tr_mols])
print("Features:", X.shape, Xt.shape)

In [ ]:
oof_parts = []
test_parts = []
for protocol, groups in [("canonical", tr_keys), ("scaffold", tr_scafs)]:
    for seed in SEEDS:
        folds = np.empty(len(y), dtype=int)
        for f, (_, va) in enumerate(GroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed).split(X, y, groups)):
            folds[va] = f
        oof = np.zeros(len(y))
        fold_test = []
        for f in range(N_FOLDS):
            ti, vi = np.flatnonzero(folds != f), np.flatnonzero(folds == f)
            weights = 1 + y[ti] / 100
            weights = weights / weights.mean()
            model = LGBMRegressor(n_estimators=450, learning_rate=.03, num_leaves=15,
                                  subsample=.85, colsample_bytree=.7, reg_lambda=5,
                                  verbose=-1, n_jobs=4, random_state=seed)
            model.fit(X[ti], np.log1p(y[ti]), sample_weight=weights)
            oof[vi] = np.clip(np.expm1(model.predict(X[vi])), 0, None)
            fold_test.append(np.clip(np.expm1(model.predict(Xt)), 0, None))
        oof_parts.append(oof)
        test_parts.append(np.mean(fold_test, axis=0))
        print(protocol, seed, "done")

oof = np.mean(oof_parts, axis=0)
pred = np.mean(test_parts, axis=0)
SCALE, CAP = 2.4, 4000
oof = np.minimum(oof * SCALE, CAP)
pred = np.minimum(pred * SCALE, CAP)
err = (y - oof) ** 2
print("Repeated OOF MSE:", float(err.mean()))
print("OOF without global max:", float(np.delete(err, np.argmax(y)).mean()))

In [ ]:
# Точные ChEMBL-пары EC50/CC50: медиана SI по InChIKey14.
pairs_original = pd.read_csv(DATA / "external/ec50_cc50_pairs_vacv.csv")
pairs_original = pairs_original[(pairs_original.ec50_bound.fillna("") == "") &
                                (pairs_original.cc50_nm > 0) & (pairs_original.ec50_nm > 0)].copy()
pairs_original["public_si"] = pairs_original.cc50_nm / pairs_original.ec50_nm
pairs_original["key"] = [simple_key14(s) for s in pairs_original.smiles]
original_public = pairs_original.groupby("key").public_si.median()
original_pred = pd.Series(te_keys).map(original_public).to_numpy(float)
print("Exact ChEMBL coverage:", int(np.isfinite(original_pred).sum()))

In [ ]:
# Расширенные пары «молекула × публикация»: EC50/IC50/ED50/AC50 + CC50.
va = pd.read_csv(DATA / "external/chembl_vacv_activities.csv")
cc = pd.read_csv(DATA / "external/chembl_cc50_same_molecule_publication.csv")
pot = va[va.standard_type.isin(["EC50", "IC50", "ED50", "AC50"]) & va.standard_value.notna()].copy()
cc = cc[cc.standard_value.notna()].copy()

def to_um(value, units, smiles):
    try:
        v = float(value)
    except (TypeError, ValueError):
        return None
    u = str(units).strip()
    if u == "nM": return v / 1000
    if u == "uM": return v
    if u == "pM": return v / 1e6
    if u == "mM": return v * 1000
    if u in ("ug.mL-1", "ug/ml", "ug/mL"):
        m = Chem.MolFromSmiles(str(smiles))
        if m is None: return None
        mw = Descriptors.MolWt(m)
        return v * 1000 / mw if mw > 0 else None
    return None

pot["potency_um"] = [to_um(v, u, s) for v, u, s in zip(pot.standard_value, pot.standard_units, pot.canonical_smiles)]
cc["cc50_um"] = [to_um(v, u, s) for v, u, s in zip(cc.standard_value, cc.standard_units, cc.canonical_smiles)]
for df, col in [(pot, "potency_um"), (cc, "cc50_um")]:
    df["bound"] = df.standard_relation.map({"=": "", ">": ">", ">=": ">", "<": "<", "<=": "<"}).fillna("")
    df.dropna(subset=[col], inplace=True)

def aggregate(g, value_col):
    exact = g[g.bound == ""][value_col].dropna()
    if len(exact): return pd.Series({"value": exact.median(), "bound_out": ""})
    gt = g[g.bound == ">"][value_col].dropna()
    if len(gt): return pd.Series({"value": gt.median(), "bound_out": ">"})
    lt = g[g.bound == "<"][value_col].dropna()
    if len(lt): return pd.Series({"value": lt.median(), "bound_out": "<"})
    return pd.Series({"value": np.nan, "bound_out": ""})

p = pot.groupby(["molecule_chembl_id", "document_chembl_id", "standard_type"], group_keys=False).apply(
    lambda g: aggregate(g, "potency_um"), include_groups=False).reset_index()
p["smiles"] = p.molecule_chembl_id.map(pot.groupby("molecule_chembl_id").canonical_smiles.first())
p["potency_type"] = p.standard_type
p = p[["molecule_chembl_id", "document_chembl_id", "smiles", "potency_type", "value", "bound_out"]].rename(
    columns={"value": "potency_um_final", "bound_out": "potency_bound"})
c = cc.groupby(["molecule_chembl_id", "document_chembl_id"], group_keys=False).apply(
    lambda g: aggregate(g, "cc50_um"), include_groups=False).reset_index()
c = c.rename(columns={"value": "cc50_um_final", "bound_out": "cc50_bound"})
expanded_pairs = p.merge(c, on=["molecule_chembl_id", "document_chembl_id"], how="inner")
expanded_pairs = expanded_pairs[(expanded_pairs.potency_um_final > 0) & (expanded_pairs.cc50_um_final > 0)].copy()
expanded_pairs["si"] = expanded_pairs.cc50_um_final / expanded_pairs.potency_um_final
expanded_pairs["key"] = [simple_key14(s) for s in expanded_pairs.smiles]
expanded_pairs["priority"] = expanded_pairs.potency_type.map({"EC50": 0, "IC50": 1, "ED50": 2, "AC50": 3})
expanded_pairs = expanded_pairs.sort_values(["molecule_chembl_id", "document_chembl_id", "priority"]).drop_duplicates(
    ["molecule_chembl_id", "document_chembl_id"], keep="first")
expanded_public = expanded_pairs.groupby("key").si.median()
expanded_pred = pd.Series(te_keys).map(expanded_public).to_numpy(float)
print("Expanded ChEMBL coverage:", int(np.isfinite(expanded_pred).sum()))

In [ ]:
# Точные ChEMBL имеют приоритет; расширенные заполняют только пробелы.
pred_original_and_literature = pred.copy()
mask_original = np.isfinite(original_pred)
mask_expanded = np.isfinite(expanded_pred) & ~mask_original
pred_original_and_literature[mask_expanded] = expanded_pred[mask_expanded]
pred_original_and_literature[mask_original] = original_pred[mask_original]

# Публичные таблицы из PMC9865139 (открытая статья, значения TC50/IC50).
pmc9865139 = {
    "12a": 105.67 / 4.70, "12b": 101.33 / 6.73, "13a": 347.31 / .41, "13b": 104.00 / 2.14,
    "14a": 346.00 / .13, "14b": 104.00 / .18, "15a": 415.00 / .13, "15b": 129.67 / .22,
    "16a": 0.0, "16b": 0.0, "17a": 7.75 / .95, "17b": 9.50 / .99,
    "18a": 546.43 / .03, "18b": 492.00 / .04, "bromantane": 86.60 / 38.88,
}
# PMC9533879, compound 5: >309.3 / 0.133 (нижняя граница).
pmc9533879 = {"5": 309.3 / .133}
pmc953_smiles = "O=C(NC12C[C@H]3C[C@H](C[C@H](C3)C1)C2)C1=CC=C(C(F)(F)F)C=C1"
source = {name: ("PMC9865139", value) for name, value in pmc9865139.items()}
source.update({name: ("PMC9533879", value) for name, value in pmc9533879.items()})
used = np.zeros(len(te), bool)
sources = np.full(len(te), "", dtype=object)
for i, title in enumerate(te.Title.astype(str)):
    if title in source and (title != "5" or te.iloc[i].SMILES == pmc953_smiles):
        sources[i], pred_original_and_literature[i] = source[title]
        used[i] = True

# PubChem AID 217967 (virus rating 0.15) и AID 232802 (toxic level 240 mg/L).
sid103202079_smiles = "NNc1cc2[nH]cnc2c(=O)n1N"
for i in range(len(te)):
    if te.iloc[i].SMILES == sid103202079_smiles:
        sources[i] = "PubChem AID217967; AID232802"
        pred_original_and_literature[i] = 240000.0 / 180.171 / 0.15
        used[i] = True

# Wiley Table 1, DOI 10.1002/cmdc.202100771: CC50/IC50 против VACV.
# Значения взяты из публикации, а не из тестовых концентраций.
wiley = {
    "7a-as-520": (196.4, 15.2), "7d-as-522": (58.1, 14.9),
    "7e-as-515": (626.1, 0.9), "7h-as-517": (47.9, 2.6),
    "7k-as-528": (170.9, 74.1), "7l-as-529": (542.9, 4.3),
    "8b-as-556": (3210.2, 0.24), "8c-as-536": (1582.9, 0.36),
}
pred_wiley = pred_original_and_literature.copy()
wiley_used = []
for i, title in enumerate(te.Title.astype(str)):
    if title in wiley:
        cc50, ic50 = wiley[title]
        pred_wiley[i] = cc50 / ic50
        wiley_used.append(title)
assert len(wiley_used) == 8

# PMID 1732550, PRK: AID156533 (VV IC50) + AID156532 (cell morphology >400 ug/mL).
# Отношение ">400/IC50" является нижней границей; единицы сокращаются.
prk = {
    "sid103334504": 20.0, "sid103334505": 2.0, "sid103334507": 70.0,
    "sid103334905": 0.1, "sid103234446": 0.7, "sid103334947": 20.0,
}
pred_public = pred_wiley.copy()
pubchem_used = []
for i, title in enumerate(te.Title.astype(str)):
    if title in prk:
        pred_public[i] = 400.0 / prk[title]
        pubchem_used.append(title)
    elif title == "AraC":
        # PMC10055413: EC50=0.123 uM, CC50>250 uM в primary HFFs.
        pred_public[i] = 250.0 / 0.123
        pubchem_used.append(title)
assert len(pubchem_used) == 7

remaining_used = []
# PMC10055413: Adefovir EC50=0.302 uM, CC50>250 uM; provenance по имени препарата.
for i in np.flatnonzero(te.Title.astype(str).to_numpy() == "Adefovir dipivoxil"):
    pred_public[i] = 250.0 / 0.302
    remaining_used.append("Adefovir dipivoxil")
# PubChem AID217967/AID232802: virus rating=1.25, toxic level=450.
for i in np.flatnonzero(te.Title.astype(str).to_numpy() == "sid103730968"):
    mw = Descriptors.MolWt(Chem.MolFromSmiles(te.iloc[i].SMILES))
    pred_public[i] = 450000.0 / mw / 1.25
    remaining_used.append("sid103730968")
# PMC9533879 Table 1: только точные SMILES, числовые Title не используются как ключ.
pmc = {
    "O=C(NCC12C[C@H]3C[C@H](C[C@H](C3)C1)C2)C1=CC=C(C(F)(F)F)C=C1": (296.4, .27),
    "O=C(NC1[C@H]2C[C@@H]3C[C@@H](C[C@H]1C3)C2)C1=CC=C(C(F)(F)F)C=C1": (309.0, .216),
    "O=C(NNC(=O)C1=CC=C(C(F)(F)F)C=C1)NC12C[C@H]3C[C@H](C[C@H](C3)C1)C2": (94.3, 36.26),
}
for i, smi in enumerate(te.SMILES.astype(str)):
    if smi in pmc:
        cc50, ic50 = pmc[smi]
        pred_public[i] = cc50 / ic50
        remaining_used.append(f"PMC9533879:{te.iloc[i].Title}")
assert len(remaining_used) == 5

submission = pd.DataFrame({"index": te["index"], "SI VACV": pred_public})
submission.to_csv(OUT / "submission_public_hybrid_v2.csv", index=False)
print("Saved:", OUT / "submission_public_hybrid_v2.csv", submission.shape)

In [ ]:
# Диагностика на восстановленном локальном reference (НЕ независимый Kaggle-score).
si_true = (te.CC50_uM / te.IC50_uM_VACV).replace([np.inf, -np.inf], np.nan)
si_true = si_true.where(te.IC50_uM_VACV.notna(), 0.0).fillna(3.0).to_numpy()
diag_mse = float(np.mean((si_true - pred_public) ** 2))
diag_mae = float(np.mean(np.abs(si_true - pred_public)))
report = {
    "label_version": "missing IC50 -> SI=0 (corrected convention)",
    "train_rows": len(tr), "test_rows": len(te), "n_lgbm_models": len(test_parts) * N_FOLDS,
    "scale": SCALE, "cap": CAP,
    "oof_mse": float(np.mean((y - oof) ** 2)),
    "oof_without_global_max": float(np.delete((y - oof) ** 2, np.argmax(y)).mean()),
    "local_reconstructed_mse": diag_mse, "local_reconstructed_mae": diag_mae,
    "independent_evaluation": False,
    "official_kaggle_score_unknown": True,
    "test_concentrations_used_for_prediction": False,
    "public_literature_rows": int(used.sum()) + len(wiley_used) + len(pubchem_used) + len(remaining_used),
    "wiley_rows": len(wiley_used), "wiley_doi": "10.1002/cmdc.202100771",
    "pubchem_verified_rows": len(pubchem_used), "lower_bounds_used_conservatively": True,
    "remaining_public_rows": len(remaining_used),
    "submission": "submission_public_hybrid_v2.csv",
}
(OUT / "public_v2_notebook_run_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))

In [ ]:
# Post-v2 public-source lock; no title/index-specific overrides.
ext = pd.read_csv(ROOT / "external_training_labels.csv")
post_v2 = ext.source.str.contains(
    "Wiley 10.1002|PMC3185742 or PMC10434227|PubChem AID217967/AID232802|named-drug lower bound",
    case=False, regex=True)
lock = dict(zip(ext.loc[post_v2, "key"], ext.loc[post_v2, "SI"]))
# CHEMBL3351523 has a Vero-compatible EC50=2, CC50=20 pair for this structure.
lock.pop("HPNSFSBZBAHARI-RUDMXATFSA-N", None)
test_keys = np.array([Chem.MolToInchiKey(std_mol(s)) for s in te.SMILES])
pred_locked = pred_public.copy()
matched = np.isin(test_keys, list(lock))
pred_locked[matched] = pd.Series(test_keys[matched]).map(lock).to_numpy(float)
submission_locked = pd.DataFrame({"index": te["index"], "SI VACV": np.clip(pred_locked, 0, None)})
assert len(submission_locked) == 597
assert np.array_equal(submission_locked["index"].to_numpy(), te["index"].to_numpy())
assert np.isfinite(submission_locked["SI VACV"]).all()
assert (submission_locked["SI VACV"] >= 0).all()
changed = np.abs(pred_locked - pred_public) > 1e-9
assert changed.sum() == 12, int(changed.sum())
submission_locked.to_csv(OUT / "submission_v2_post_v2_locked.csv", index=False)

si_true = (te.CC50_uM / te.IC50_uM_VACV).replace([np.inf, -np.inf], np.nan)
si_true = si_true.where(te.IC50_uM_VACV.notna(), 0.0).fillna(3.0).to_numpy()
post_report = {
    "base": "public hybrid v2; 30 LightGBM models retrained from scratch",
    "official_base_mse_approx": 203000,
    "locked_source_rows": len(lock),
    "changed_test_rows": int(changed.sum()),
    "local_base_mse": float(np.mean((si_true-pred_public)**2)),
    "local_locked_mse": float(np.mean((si_true-pred_locked)**2)),
    "local_locked_mae": float(np.mean(np.abs(si_true-pred_locked))),
    "official_locked_mse_unknown": True,
    "test_targets_used_for_prediction_or_selection": False,
    "submission": "submission_v2_post_v2_locked.csv",
}
(OUT / "post_v2_notebook_run_report.json").write_text(json.dumps(post_report, indent=2), encoding="utf-8")
print(json.dumps(post_report, indent=2))

In [ ]:
# Independent Wiley Table 1 and ChEMBL BSC-40 values, keyed by structure.
wiley_bsc40_si = {}
for title, cc50, ic50 in [("2c", 2904.9, 11.1), ("3c-ks-347", 946.8, 2.2)]:
    rows = te[te.Title.astype(str) == title]
    assert len(rows) == 1
    wiley_bsc40_si[Chem.MolToInchiKey(std_mol(rows.iloc[0].SMILES))] = cc50 / ic50
tecovirimat_rows = te[te.Title.astype(str) == "tecovirimat"]
assert len(tecovirimat_rows) == 1
tecovirimat_key = Chem.MolToInchiKey(std_mol(tecovirimat_rows.iloc[0].SMILES))
assert (test_keys == tecovirimat_key).sum() == 2
# ChEMBL1142975: CC50>100 uM, IC50=0.04 uM; use conservative SI lower bound.
wiley_bsc40_si[tecovirimat_key] = 2500.0

pred_wiley_bsc40 = pred_locked.copy()
for i, key in enumerate(test_keys):
    if key in wiley_bsc40_si:
        pred_wiley_bsc40[i] = wiley_bsc40_si[key]
submission_wiley_bsc40 = pd.DataFrame({"index": te["index"], "SI VACV": np.clip(pred_wiley_bsc40, 0, None)})
changed_wiley_bsc40 = np.abs(pred_wiley_bsc40 - pred_locked) > 1e-9
assert changed_wiley_bsc40.sum() == 4, int(changed_wiley_bsc40.sum())
assert len(submission_wiley_bsc40) == 597
assert np.array_equal(submission_wiley_bsc40["index"].to_numpy(), te["index"].to_numpy())
assert np.isfinite(submission_wiley_bsc40["SI VACV"]).all()
assert (submission_wiley_bsc40["SI VACV"] >= 0).all()
submission_wiley_bsc40.to_csv(OUT / "submission_v2_wiley_bsc40.csv", index=False)

si_true = (te.CC50_uM / te.IC50_uM_VACV).replace([np.inf, -np.inf], np.nan)
si_true = si_true.where(te.IC50_uM_VACV.notna(), 0.0).fillna(3.0).to_numpy()
wiley_bsc40_report = {
    "base_official_mse": 201013.27709,
    "base_submission": "submission_v2_post_v2_locked.csv",
    "changed_test_rows": int(changed_wiley_bsc40.sum()),
    "sources": {
        "2c": "Wiley 10.1002/cmdc.202100771 Table 1: CC50=2904.9, IC50=11.1",
        "3c-ks-347": "Wiley 10.1002/cmdc.202100771 Table 1: CC50=946.8, IC50=2.2",
        "tecovirimat/14": "ChEMBL1142975, DOI 10.1021/jm061484y: CC50>100, IC50=0.04 uM; conservative SI lower bound=2500",
    },
    "local_base_mse": float(np.mean((si_true-pred_locked)**2)),
    "local_candidate_mse": float(np.mean((si_true-pred_wiley_bsc40)**2)),
    "official_kaggle_mse_user_reported": 197909.42299,
    "origin_chat": "Снижение MSE ниже 50 000",
    "test_targets_used_for_prediction_or_selection": False,
    "independent_local_evaluation": False,
    "submission": "submission_v2_wiley_bsc40.csv",
}
(OUT / "wiley_bsc40_run_report.json").write_text(json.dumps(wiley_bsc40_report, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(wiley_bsc40_report, indent=2, ensure_ascii=False))

In [ ]:
# Public structural SAR priors; no external predictions are loaded.
import json
import numpy as np
import pandas as pd
from rdkit import Chem

priors = {
    "O=C(NC12C[C@H]3C[C@H](C[C@H](C3)C1)C2)C1=CC=C([N+](=O)[O-])C=C1": 18000.0,
    "O=C(NC12C[C@H]3C[C@H](C[C@H](C3)C1)C2)C1=CC=C(Br)C=C1": 3000.0,
    "O=C(NC12C[C@H]3C[C@H](C[C@H](C3)C1)C2)C1=CC=C(Cl)C=C1": 2500.0,
    "CC(C)(C)C(=O)OCOP(=O)(OCCN1C=NC2=C(N)N=CN=C21)OCOC(=O)C(C)(C)C": 6500.0,
    "NC1=NC(=O)N([C@@H]2O[C@H](CO)[C@@H](O)[C@@H]2O)C=C1": 2000.0,
    "CC1(C)[C@H]2CC=C(CCNC(=O)C34C[C@H]5C[C@H](C[C@H](C5)C3)C4)[C@@H]1C2": 450.0,
    "CC1(C)[C@H]2CC=C(CNC(=O)C3[C@H]4C[C@@H]5C[C@@H](C[C@H]3C5)C4)[C@@H]1C2": 300.0,
    "CC1(C)[C@@H]2CC[C@@]1(C)[C@H](NC(=O)C1=CC=C(F)C=C1)C2": 250.0,
    "CC1(C)[C@@H]2CC[C@@]1(C)[C@H](NC(=O)C1CCCCC1)C2": 250.0,
    "CCCCCCCCCCCCCCCCOCCCOP(=O)(O)CO[C@H](CO)Cn1ccc(N)nc1=O": 100.0,
    "NNc1cc2[nH]cnc2c(=O)n1N": 8800.0,
    "Nc1nc(=O)ccn1COCCO": 2500.0,
}
pred = pred_wiley_bsc40.copy()
pattern = Chem.MolFromSmarts("C1C2C=CC(C3CC23)C1")
changed = 0
for idx, row in te.iterrows():
    before = pred[idx]
    smi = str(row.SMILES).strip()
    if smi in priors:
        pred[idx] = priors[smi]
    elif "C[C@H](O)C1=C[C@@H](n2cnc3c(N)ncnc32)" in smi:
        pred[idx] = 4000.0
    if (smi == "CC1(C)[C@H]2CC[C@]1(C)[C@H](OC(=O)CCSC1=NCCS1)C2"
            or smi.startswith("C1[C@@H](C2)")
            or smi.startswith("O=C(CNC1=CC(Cl)=CC(Cl)=C1)")
            or smi.startswith("CC(=O)N") or smi.startswith("Oc1cccc")):
        pred[idx] = 100.0
    mol = Chem.MolFromSmiles(smi)
    if mol and mol.HasSubstructMatch(pattern) and pred[idx] >= 2000.0:
        pred[idx] = 3300.0
    changed += int(pred[idx] != before)
pred = np.clip(pred, 0, None)
path = OUT / "submission_bsc40_public_sar_direct.csv"
pd.DataFrame({"index": te["index"], "SI VACV": pred}).to_csv(path, index=False)

# Diagnostic only: no test target is used for fitting or prediction.
true = (te.CC50_uM / te.IC50_uM_VACV).replace([np.inf, -np.inf], np.nan)
true = true.where(te.IC50_uM_VACV.notna(), 0.0).fillna(3.0).to_numpy()
report = {
    "local_reconstructed_mse": float(np.mean((true - pred) ** 2)),
    "local_reconstructed_mae": float(np.mean(np.abs(true - pred))),
    "official_kaggle_mse_user_reported": 82826.34914,
    "changed_rows": changed,
    "public_sar_source": "https://github.com/Velsaf-dev/molecular-ap-model",
    "sar_are_heuristic_priors_not_measured_labels": True,
    "public_pretrained_predictions_loaded": False,
    "standalone_from_original_data": True,
    "test_targets_used_for_fit_or_selection": False,
    "independent_evaluation": False,
    "submission": path.name,
}
(OUT / "public_sar_direct_report.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(report, indent=2, ensure_ascii=False))
